In [20]:
from langchain_text_splitters         import RecursiveCharacterTextSplitter
from langchain_core.prompts           import ChatPromptTemplate
from langchain_core.documents         import Document
from langchain_core.output_parsers    import StrOutputParser
from langchain_core.runnables         import RunnablePassthrough, RunnableParallel
from langchain_anthropic              import ChatAnthropic
from langchain_community.vectorstores import FAISS

# Tenta o import moderno primeiro, cai para o legado se necessário
try:
    from langchain_huggingface import HuggingFaceEmbeddings
    print("✅ HuggingFaceEmbeddings → langchain_huggingface")
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings
    print("✅ HuggingFaceEmbeddings → langchain_community (legado)")

import pandas as pd
import json
import logging
import re
from datetime import datetime
from pathlib import Path

print("✅ Todos os imports OK!")

✅ HuggingFaceEmbeddings → langchain_community (legado)
✅ Todos os imports OK!


In [21]:
Path("../logs").mkdir(exist_ok=True)

logging.basicConfig(
    level    = logging.INFO,
    format   = '%(asctime)s | %(levelname)s | %(message)s',
    handlers = [
        logging.FileHandler(
            f"../logs/assistente_{datetime.now().strftime('%Y%m%d')}.log",
            encoding='utf-8'
        ),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("AssistenteMedico")

logger.info("="*60)
logger.info("ASSISTENTE MÉDICO — TECH CHALLENGE FASE 3")
logger.info("="*60)
logger.info(f"Sessão iniciada: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

print("✅ Sistema de logging configurado!")
print("   → Logs salvos em: ../logs/")

✅ Sistema de logging configurado!
   → Logs salvos em: ../logs/


In [22]:
df = pd.read_csv("../data/processed/dados_medicos_rag.csv")
print(f"✅ {len(df)} registros carregados")

documentos = []
for _, row in df.iterrows():
    conteudo = f"Pergunta: {row['input']}\nResposta: {row['output']}"
    doc = Document(
        page_content = conteudo,
        metadata     = {
            "source":   row['source'],
            "pergunta": str(row['input'])[:100],
        }
    )
    documentos.append(doc)

splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 1000,
    chunk_overlap = 100,
)
chunks = splitter.split_documents(documentos)

logger.info(f"Base de conhecimento: {len(documentos)} docs → {len(chunks)} chunks")
print(f"✅ {len(documentos)} documentos → {len(chunks)} chunks")

✅ 1000 registros carregados
✅ 1000 documentos → 1439 chunks


In [23]:
print("⏳ Carregando modelo de embeddings médicos...")
print("   (Primeira vez demora ~2 minutos para baixar)\n")

embeddings = HuggingFaceEmbeddings(
    model_name    = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb",
    model_kwargs  = {"device": "cpu"},
    encode_kwargs = {"normalize_embeddings": True},
)

print("⏳ Indexando documentos no FAISS...")
vectorstore = FAISS.from_documents(chunks, embeddings)

Path("../data/processed/vectorstore").mkdir(parents=True, exist_ok=True)
vectorstore.save_local("../data/processed/vectorstore")

logger.info(f"Vector store criado: {len(chunks)} vetores indexados")
print(f"\n✅ Vector store criado e salvo!")
print(f"   {len(chunks)} vetores indexados")

# Teste de busca
print("\n🔍 Teste de busca:")
query      = "fatores de risco câncer de mama"
resultados = vectorstore.similarity_search(query, k=2)
for i, r in enumerate(resultados):
    print(f"\n  Resultado {i+1} [{r.metadata['source']}]:")
    print(f"  {r.page_content[:200]}...")

/tmp/ipykernel_13467/3346753841.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
[huggingface_hub.utils._http|WARNING]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


⏳ Carregando modelo de embeddings médicos...
   (Primeira vez demora ~2 minutos para baixar)



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 46686.79it/s]
BertModel LOAD REPORT from: pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳ Indexando documentos no FAISS...

✅ Vector store criado e salvo!
   1439 vetores indexados

🔍 Teste de busca:

  Resultado 1 [PubMedQA]:
  Pergunta: Risk factors for major depression during midlife among a community sample of women with and without prior major depression: are they the same or different?
Resposta: The menopausal transitio...

  Resultado 2 [MedQuAD]:
  Pergunta: What are the treatments for Mabry syndrome ?
Resposta: These resources address the diagnosis or management of Mabry syndrome:  - Genetic Testing Registry: Hyperphosphatasia with mental retar...


In [28]:
from dotenv import load_dotenv
import os

load_dotenv()  # carrega o .env automaticamente

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

if not ANTHROPIC_API_KEY:
    raise ValueError(
        "❌ ANTHROPIC_API_KEY não encontrada!\n"
        "   Crie um arquivo .env na raiz do projeto com:\n"
        "   ANTHROPIC_API_KEY=sua-chave-aqui"
    )

print("✅ Chave carregada com segurança via variável de ambiente")

llm = ChatAnthropic(
    model       = "claude-sonnet-4-6",
    api_key     = ANTHROPIC_API_KEY,
    temperature = 0.2,
    max_tokens  = 1024,
)

PROMPT_MEDICO = ChatPromptTemplate.from_template("""
Você é um assistente médico de apoio clínico desenvolvido para
auxiliar profissionais de saúde. Suas respostas são baseadas em
protocolos médicos e evidências científicas.

REGRAS OBRIGATÓRIAS:
1. Responda SEMPRE em português brasileiro
2. NUNCA faça prescrições diretas — sugira avaliação médica
3. Cite SEMPRE a fonte do conhecimento utilizado
4. Se não souber, diga claramente que não tem informação suficiente
5. Lembre que o médico sempre tem a palavra final

CONTEXTO MÉDICO RECUPERADO:
{context}

PERGUNTA DO PROFISSIONAL DE SAÚDE:
{input}

RESPOSTA ESTRUTURADA:
""")

logger.info("LLM configurada: claude-sonnet-4-6 | temp=0.2")
print("✅ Claude configurado como LLM principal!")
print("   Temperatura: 0.2 (conservador para contexto médico)")

✅ Chave carregada com segurança via variável de ambiente
✅ Claude configurado como LLM principal!
   Temperatura: 0.2 (conservador para contexto médico)


In [25]:
def formatar_docs(docs):
    """Concatena os documentos recuperados em um único bloco de contexto."""
    return "\n\n---\n\n".join(
        f"[Fonte: {doc.metadata.get('source', 'N/A')}]\n{doc.page_content}"
        for doc in docs
    )

retriever = vectorstore.as_retriever(
    search_type   = "similarity",
    search_kwargs = {"k": 4},
)

# Pipeline LCEL:
# RunnableParallel roda em paralelo:
#   → context: busca docs no FAISS e formata
#   → input:   passa a pergunta direto para o prompt
# Depois encadeia: Prompt → Claude → Parser de texto
chain_rag = (
    RunnableParallel({
        "context": retriever | formatar_docs,
        "input":   RunnablePassthrough(),
    })
    | PROMPT_MEDICO
    | llm
    | StrOutputParser()
)

logger.info("Pipeline RAG configurado: LCEL + FAISS + Claude")
print("✅ Pipeline RAG montado com LCEL!")
print("   Fluxo: input → FAISS | passthrough → Prompt → Claude → Resposta")

✅ Pipeline RAG montado com LCEL!
   Fluxo: input → FAISS | passthrough → Prompt → Claude → Resposta


In [26]:
TOPICOS_BLOQUEADOS = [
    "prescrever", "receitar", "posologia exata",
    "dose de", "mg por kg", "autom[eé]dicar",
    "sem consultar", "diagnóstico definitivo"
]

def assistente_medico(pergunta: str, id_profissional: str = "medico_01") -> dict:
    inicio = datetime.now()
    logger.info(f"[{id_profissional}] Nova consulta: {pergunta[:80]}...")

    # Validação de segurança
    for topico in TOPICOS_BLOQUEADOS:
        if re.search(topico, pergunta.lower()):
            logger.warning(f"[{id_profissional}] Bloqueado: '{topico}' detectado")
            return {
                "resposta" : (
                    "⚠️ Esta consulta envolve prescrição ou diagnóstico definitivo, "
                    "fora do escopo deste assistente. Consulte um médico especialista."
                ),
                "fontes"   : [],
                "bloqueado": True,
                "tempo_ms" : 0,
            }

    try:
        resposta         = chain_rag.invoke(pergunta)
        docs_recuperados = retriever.invoke(pergunta)

        fontes = [
            {
                "source": doc.metadata.get("source", "N/A"),
                "trecho": doc.page_content[:150] + "...",
            }
            for doc in docs_recuperados
        ]

        tempo_ms = int((datetime.now() - inicio).total_seconds() * 1000)
        logger.info(
            f"[{id_profissional}] Resposta em {tempo_ms}ms | "
            f"Fontes: {[f['source'] for f in fontes]}"
        )

        return {
            "pergunta" : pergunta,
            "resposta" : resposta,
            "fontes"   : fontes,
            "bloqueado": False,
            "tempo_ms" : tempo_ms,
        }

    except Exception as e:
        logger.error(f"[{id_profissional}] Erro: {str(e)}")
        raise

print("✅ Função do assistente médico configurada!")
print("   → Validação de tópicos bloqueados")
print("   → Logging completo com ID do profissional")
print("   → Citação automática de fontes")
print("   → Medição de tempo de resposta")

✅ Função do assistente médico configurada!
   → Validação de tópicos bloqueados
   → Logging completo com ID do profissional
   → Citação automática de fontes
   → Medição de tempo de resposta


In [27]:
consultas = [
    ("Quais são os critérios diagnósticos para diabetes tipo 2?",        "dr_silva"),
    ("Como identificar sinais precoces de câncer de mama?",              "dra_costa"),
    ("Quais exames são recomendados para rastreamento cardiovascular?",  "dr_lima"),
    ("Me dê uma receita de metformina 500mg para minha paciente.",       "dr_silva"),
]

print("🏥 TESTES DO ASSISTENTE MÉDICO COM RAG\n")
print("="*65)

for pergunta, medico in consultas:
    print(f"\n👨‍⚕️ [{medico}] {pergunta}")
    resultado = assistente_medico(pergunta, medico)

    if resultado["bloqueado"]:
        print(f"\n🚫 BLOQUEADO: {resultado['resposta']}")
    else:
        print(f"\n🤖 Resposta:\n{resultado['resposta']}")
        print(f"\n📚 Fontes utilizadas:")
        for f in resultado["fontes"][:2]:
            print(f"   [{f['source']}] {f['trecho']}")
        print(f"\n⏱  Tempo: {resultado['tempo_ms']}ms")

    print("="*65)

🏥 TESTES DO ASSISTENTE MÉDICO COM RAG


👨‍⚕️ [dr_silva] Quais são os critérios diagnósticos para diabetes tipo 2?

🤖 Resposta:
# 🩺 Critérios Diagnósticos para Diabetes Mellitus Tipo 2

---

## ⚠️ Aviso Importante
> As informações abaixo são de apoio clínico baseadas em diretrizes reconhecidas. **O médico responsável sempre tem a palavra final** no diagnóstico e conduta clínica.

---

## 📋 Critérios Diagnósticos Estabelecidos

De acordo com a **American Diabetes Association (ADA, Standards of Medical Care in Diabetes – 2024)** e referendados pela **Sociedade Brasileira de Diabetes (SBD, Diretriz 2023-2024)**, o diagnóstico de DM tipo 2 pode ser estabelecido por **qualquer um** dos seguintes critérios:

| Exame | Valor de Corte | Observação |
|-------|---------------|------------|
| **Glicemia de jejum** | ≥ 126 mg/dL | Jejum mínimo de 8 horas |
| **Glicemia 2h pós-TOTG** | ≥ 200 mg/dL | Carga de 75g de glicose anidra |
| **Hemoglobina glicada (HbA1c)** | ≥ 6,5% | Método certificado pelo